## RAG WITH MULTI DATA SOURCE


## 1.wikipedia tool

In [1]:
## wikipedia tool
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper


C:\Users\enggm\AppData\Local\Temp\ipykernel_22152\22798755.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import WikipediaQueryRun


In [2]:
## initialization
api_wrapper=WikipediaAPIWrapper(top_k_results=1,doc_content_chars_max=250)
wiki=WikipediaQueryRun(api_wrapper=api_wrapper)
print(wiki.name)


wikipedia


In [3]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_ollama import OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader=WebBaseLoader("https://docs.smith.langchain.com/")
docs=loader.load()
documents=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200).split_documents(docs)
vectordb = FAISS.from_documents(
    documents,
    OllamaEmbeddings(model="nomic-embed-text")
)
retriever = vectordb.as_retriever() ## in-builder retriver tool
retriever


USER_AGENT environment variable not set, consider setting it to identify your requests.


VectorStoreRetriever(tags=['FAISS', 'OllamaEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x00000284CFE9BCB0>, search_kwargs={})

## 2.Custom tool

In [4]:
## custom tool
from langchain_core.tools.retriever import create_retriever_tool

retriever_tool = create_retriever_tool(
    retriever,
    "langsmith_search",
    "Search for information about LangSmith. Use this tool when user asks about LangSmith or LangChain docs."
)

retriever_tool

StructuredTool(name='langsmith_search', description='Search for information about LangSmith. Use this tool when user asks about LangSmith or LangChain docs.', args_schema=<class 'langchain_core.tools.retriever.RetrieverInput'>, func=<function create_retriever_tool.<locals>.func at 0x00000284D1C3F530>, coroutine=<function create_retriever_tool.<locals>.afunc at 0x00000284FC826FB0>)

In [5]:
retriever_tool.name

'langsmith_search'

## 3.Arxix tool

In [6]:
## arxiv tool
from langchain_community.tools import ArxivQueryRun
from langchain_community.utilities import ArxivAPIWrapper 


In [7]:
## initialization
arxiv_wrapper=ArxivAPIWrapper(top_k_results=1,doc_content_chars_max=250)
arxiv=ArxivQueryRun(arxiv_wrapper=arxiv_wrapper)
arxiv.name

'arxiv'

### step 1 => COMBINING TOOLS

In [8]:
tools = [wiki,arxiv,retriever_tool]

print(tools)

[WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from 'e:\\Machine learning\\Langchain\\.venv\\Lib\\site-packages\\wikipedia\\__init__.py'>, top_k_results=1, lang='en', load_all_available_meta=False, doc_content_chars_max=250)), ArxivQueryRun(api_wrapper=ArxivAPIWrapper(arxiv_search=<class 'arxiv.Search'>, arxiv_exceptions=(<class 'arxiv.ArxivError'>, <class 'arxiv.UnexpectedEmptyPageError'>, <class 'arxiv.HTTPError'>), top_k_results=3, ARXIV_MAX_QUERY_LENGTH=300, continue_on_failure=False, load_max_docs=100, load_all_available_meta=False, doc_content_chars_max=4000)), StructuredTool(name='langsmith_search', description='Search for information about LangSmith. Use this tool when user asks about LangSmith or LangChain docs.', args_schema=<class 'langchain_core.tools.retriever.RetrieverInput'>, func=<function create_retriever_tool.<locals>.func at 0x00000284D1C3F530>, coroutine=<function create_retriever_tool.<locals>.afunc at 0x00000284FC826FB0>)]


In [9]:
## tool and print(tool)
## same output fro both
## we can print in both ways

In [10]:
tools

[WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from 'e:\\Machine learning\\Langchain\\.venv\\Lib\\site-packages\\wikipedia\\__init__.py'>, top_k_results=1, lang='en', load_all_available_meta=False, doc_content_chars_max=250)),
 ArxivQueryRun(api_wrapper=ArxivAPIWrapper(arxiv_search=<class 'arxiv.Search'>, arxiv_exceptions=(<class 'arxiv.ArxivError'>, <class 'arxiv.UnexpectedEmptyPageError'>, <class 'arxiv.HTTPError'>), top_k_results=3, ARXIV_MAX_QUERY_LENGTH=300, continue_on_failure=False, load_max_docs=100, load_all_available_meta=False, doc_content_chars_max=4000)),
 StructuredTool(name='langsmith_search', description='Search for information about LangSmith. Use this tool when user asks about LangSmith or LangChain docs.', args_schema=<class 'langchain_core.tools.retriever.RetrieverInput'>, func=<function create_retriever_tool.<locals>.func at 0x00000284D1C3F530>, coroutine=<function create_retriever_tool.<locals>.afunc at 0x00000284FC826FB0>)]

 ## step 2 => making an llm model using ollama

In [11]:
from dotenv import load_dotenv
import os
from langchain_ollama import ChatOllama

load_dotenv()

llm = ChatOllama(model="llama3", temperature=0)

 ## step 3 => creating prompt template

In [12]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant"),
    MessagesPlaceholder(variable_name="chat_history", optional=True),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

prompt.messages  

[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a helpful assistant'), additional_kwargs={}),
 MessagesPlaceholder(variable_name='chat_history', optional=True),
 HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={}),
 MessagesPlaceholder(variable_name='agent_scratchpad')]

## step 4 => Agent

In [14]:
from langchain_classic.agents import create_tool_calling_agent, AgentExecutor
from langchain_ollama import ChatOllama 
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# LLM — ChatOllama use  (tool calling support)
llm = ChatOllama(model="llama3.2")

# Prompt
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

# Agent
agent = create_tool_calling_agent(llm, tools, prompt)

# Executor
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

# Test
response = agent_executor.invoke({"input": "What is Scaled Dot Product Attention?"})
print(response["output"])



> Entering new AgentExecutor chain...

Invoking: `wikipedia` with `{'query': 'Scaled Dot Product Attention'}`


Page: Attention (machine learning)
Summary: In machine learning, attention is a method that determines the importance of each component in a sequence relative to the other components in that sequence. In natural language processing, importance is repAttention mechanisms are widely used in deep learning models, particularly in natural language processing tasks such as machine translation, question answering, and text summarization.

The Scaled Dot-Product Attention mechanism was introduced in the paper "Attention Is All You Need" by Vaswani et al. in 2017. It is a type of self-attention mechanism that allows the model to weigh the importance of different input elements relative to each other.

The scaled dot-product attention mechanism can be summarized as follows:

1. Query (Q): The query is a vector that represents the input element.
2. Key (K): The key is also a vector th

In [15]:
response = agent_executor.invoke({"input": "Tell me about Langsmith"})
print(response["output"])



> Entering new AgentExecutor chain...

Invoking: `langsmith_search` with `{'query': 'Langsmith'}`


cause, and resolve them with LangSmith Engine.For terminology and core concepts, refer to Observability concepts. For trace pricing, retention, and limits, see Usage and billing.To set up a LangSmith instance, visit the Platform setup section to choose between cloud, hybrid, or self-hosted. All options include observability, evaluation, prompt engineering, and deployment.

LangSmith Observability - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentDocs by LangChain home pageMonitorSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangSmith ObservabilityOverviewEngineTraceDebugObserveLangSmith ObservabilityLangSmith Observability provides full visibility into your LLM application: from individual traces to production-wide performance metrics.